## load config

In [6]:
import os
from dotenv import load_dotenv

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

MILVUS_URI = os.getenv("MILVUS_URI")
MILVUS_TOKEN = os.getenv("MILVUS_TOKEN")
MILVUS_COLLECTION = os.getenv(
    "MILVUS_COLLECTION",
    "property_listings",
)
API_KEY = os.getenv("ANTHROPIC_API_KEY")
MODEL = os.getenv(
    "ANTHROPIC_MODEL",
    "claude-haiku-4-5-20251001",
)

assert all([NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD]), (
    "Neo4j configuration is incomplete. Check your .env file."
)

assert MILVUS_URI, "MILVUS_URI is missing from .env"

assert API_KEY, (
    f"ANTHROPIC_API_KEY is missing. Check {ROOT / '.env'}"
)

print("Configuration loaded.")
print("Neo4j URI configured:", bool(NEO4J_URI))
print("Milvus URI configured:", bool(MILVUS_URI))
print("API key loaded:", bool(API_KEY))
print("Model configured:", MODEL)



Configuration loaded.
Neo4j URI configured: True
Milvus URI configured: True
API key loaded: True
Model configured: claude-haiku-4-5-20251001


## test Neo4j

In [3]:
from neo4j import GraphDatabase

neo4j_driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
)

neo4j_driver.verify_connectivity()

with neo4j_driver.session() as session:
    result = session.run("RETURN 1 AS test")
    print("Neo4j test:", result.single()["test"])

print("Neo4j connection successful.")

Neo4j test: 1
Neo4j connection successful.


## test Milvus

In [4]:
from pymilvus import MilvusClient

milvus_kwargs = {"uri": MILVUS_URI}

if MILVUS_TOKEN:
    milvus_kwargs["token"] = MILVUS_TOKEN

milvus_client = MilvusClient(**milvus_kwargs)

print("Milvus connection successful.")
print("Existing collections:", milvus_client.list_collections())

Milvus connection successful.
Existing collections: []


In [5]:
neo4j_driver.close()
milvus_client.close()

## test llm

In [7]:

from anthropic import Anthropic

client = Anthropic(api_key=API_KEY)

response = client.messages.create(
    model=MODEL,
    max_tokens=100,
    messages=[
        {
            "role": "user",
            "content": (
                "Reply in one sentence: "
                "I am testing Claude Haiku 4.5 for a real-estate "
                "recommendation project."
            ),
        }
    ],
)

print("Model:", response.model)
print("Stop reason:", response.stop_reason)

for block in response.content:
    if block.type == "text":
        print(block.text)


Model: claude-haiku-4-5-20251001
Stop reason: end_turn
Claude Haiku is a capable lightweight model, but I should clarify that I'm Claude 3.5 Haiku (the current version), and I can certainly help with real estate recommendation tasks like analyzing property features, market data, or generating personalized suggestions.


In [9]:
print("Input tokens:", response.usage.input_tokens)
print("Output tokens:", response.usage.output_tokens)
print("Output tokens:", response.model)



Input tokens: 32
Output tokens: 59
Output tokens: claude-haiku-4-5-20251001
